In [2]:
import geopandas as gpd
import pandas as pd
from shapely.wkt import loads as load_wkt
from shapely.geometry import Point

# Bike

In [3]:
bike_df = pd.read_csv('../citi_bike/bike_station_df.csv')
grid_df = pd.read_csv('../manhattan_grid/manhattan_grid.csv')


In [3]:
bike_df.head()

,Unnamed: 0,lon,short_name,region_id,capacity,station_id,name,lat,rental_uris.android,rental_uris.ios,geometry,index_right,zone
0,0,-73.987030,5303.06,71.0,69,66dbc420-0aca-11e7-82f6-3863bb44ef7c,Clinton St & Grand St,40.715595,https://bkn.lft.to/lastmile_qr_scan,https://bkn.lft.to/lastmile_qr_scan,POINT (-73.9870295 40.71559509),1351,Two Bridges/Seward Park
1,2,-73.973900,7502.01,71.0,24,66dde4ef-0aca-11e7-82f6-3863bb44ef7c,W 92 St & Broadway,40.792100,https://bkn.lft.to/lastmile_qr_scan,https://bkn.lft.to/lastmile_qr_scan,POINT (-73.9739 40.7921),1492,Upper West Side North
2,4,-73.930770,7327.01,71.0,18,655ad2f2-6bcc-4eee-95a9-4f528c0b81cd,Wards Meadow Comfort Station,40.782940,https://bkn.lft.to/lastmile_qr_scan,https://bkn.lft.to/lastmile_qr_scan,POINT (-73.93077 40.78294),1157,Randalls Island
3,5,-73.952060,7443.17,71.0,25,66dde76a-0aca-11e7-82f6-3863bb44ef7c,E 98 St & Park Ave,40.788130,https://bkn.lft.to/lastmile_qr_scan,https://bkn.lft.to/lastmile_qr_scan,POINT (-73.95206 40.78813),411,East Harlem South
4,6,-73.945108,8123.06,71.0,31,1859438049977968878,Broadway & W 157 St,40.834027,https://bkn.lft.to/lastmile_qr_scan,https://bkn.lft.to/lastmile_qr_scan,POINT (-73.94510813442648 40.83402740143528),1605,Washington Heights South


In [4]:
grid_df.head()

,grid_id,lat,lon,geometry
0,M-0001,40.684043,-74.025671,POLYGON ((-74.0242002152204606 40.682916945445...
1,M-0002,40.684043,-74.022730,POLYGON ((-74.0212590387498750 40.682916945445...
2,M-0003,40.684043,-74.019788,POLYGON ((-74.0183178622792894 40.682916945445...
3,M-0004,40.686295,-74.025671,POLYGON ((-74.0242002152204606 40.685169197697...
4,M-0005,40.686295,-74.022730,POLYGON ((-74.0212590387498750 40.685169197697...


In [5]:
grid_df['geometry'] = grid_df['geometry'].apply(load_wkt)

grid_gdf = gpd.GeoDataFrame(grid_df, geometry='geometry', crs='EPSG:4326')
bike_df['geometry'] = bike_df['geometry'].apply(load_wkt)
bike_gdf = gpd.GeoDataFrame(bike_df, geometry='geometry', crs='EPSG:4326')

# Drop existing 'index_right' column if it exists
if 'index_right' in bike_gdf.columns:
    bike_gdf = bike_gdf.drop(columns=['index_right'])
if 'index_right' in grid_gdf.columns:
    grid_gdf = grid_gdf.drop(columns=['index_right'])

# Spatial join: find which stations fall within which grid
joined = gpd.sjoin(bike_gdf, grid_gdf, how='inner', predicate='within')


# Get the mapping result
grid_station_summary = joined.groupby('grid_id')['station_id'].agg(['count', list]).reset_index()
grid_station_summary.columns = ['grid_id', 'station_count', 'station_ids']


# Filter grids that contain more than one station
multiple_station_bike_grids = grid_station_summary[grid_station_summary['station_count'] > 1]
multiple_station_bike_grids


,grid_id,station_count,station_ids
3,M-0038,2,"[1954806962718223342, 66dc0f1b-0aca-11e7-82f6-..."
4,M-0039,2,"[c00ef46d-fcde-48e2-afbd-0fb595fe3fa7, 66db441..."
7,M-0046,4,"[66db75cc-0aca-11e7-82f6-3863bb44ef7c, 66dbf66..."
11,M-0054,2,"[66dbca0f-0aca-11e7-82f6-3863bb44ef7c, 66db460..."
15,M-0060,2,"[3e9e50cc-f336-439f-bd0b-dec5499c038d, 6067af3..."
...,...,...,...
519,M-1062,2,"[2546c9b9-5b1a-4e9f-a9cc-f8b986b79748, 8d650b0..."
526,M-1080,2,"[e1a3f2f9-618f-4727-b2d6-676e02aadf17, 23bc4d9..."
527,M-1081,2,"[fbaffdbd-dab5-4127-9bd5-36bf58093ea8, 99b7c9a..."
529,M-1088,2,"[0e82ab9f-8c93-4302-b9df-342f8135a05c, a5bd1f5..."


In [6]:
multiple_station_bike_grids.to_csv("multiple_station_bike_grids.csv")

# Subway

In [7]:
subway_df = pd.read_csv('../mta_subway/mta_subway_20240101_20250430.csv')

In [8]:
subway_stations = subway_df[['latitude', 'longitude']].drop_duplicates()
subway_stations

,latitude,longitude
0,40.702070,-74.013664
1,40.703087,-74.012990
2,40.704820,-74.014070
3,40.706474,-74.011055
4,40.706820,-74.009100
...,...,...
3005,40.768246,-73.981926
8584,40.737335,-73.996790
15617,40.755290,-73.986755
15846,40.718803,-73.999890


In [9]:
subway_stations['geometry'] = subway_stations.apply(lambda row: Point(row['longitude'], row['latitude']), axis=1)
subway_gdf = gpd.GeoDataFrame(subway_stations, geometry='geometry', crs='EPSG:4326')

#  Spatial join — find which grid each subway station falls into
subway_with_grid = gpd.sjoin(subway_gdf, grid_gdf, how='inner', predicate='within')

# Group by grid_id and count stations
subway_station_counts = subway_with_grid.groupby('grid_id').size().reset_index(name='subway_station_count')
multiple_station_subway_grids = subway_station_counts[subway_station_counts['subway_station_count'] > 1]
print(len(multiple_station_subway_grids),"grids have multiple subway staions")
multiple_station_subway_grids

20 grids have multiple subway staions


,grid_id,subway_station_count
0,M-0038,2
2,M-0052,2
3,M-0053,2
6,M-0075,3
7,M-0076,6
9,M-0091,3
14,M-0124,3
16,M-0128,3
20,M-0139,2
25,M-0169,2
